In [2]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)
from jax_sn.operators.transport_operator import TransportOperatorBatched, TransportOperatorMultipleAnglesVmap, TransportOperatorVmap, TransportOperatorBatchedPallas, TransportOperatorMultipleAngles
from jax_sn.domain.cross_sections import CrossSectionIndexed
import equinox as eqx
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)
from finite_elements.test_moment_scatter_operator import _setup_all_sb

In [3]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain
from jax_sn.source import Source
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory
import lineax as lx

In [7]:
from finite_elements.test_breeding_blanket import bb_data_dict, base_region_sizes, base_resolutions,  _get_bb_args, _setup_bb_args, _get_bb_args


def setup_mgop(dim, degree):
    MGOP, source_eg0, solution_domain = _setup_bb_args(dim, degree, base_region_sizes(), base_resolutions(), 5.0, 10, 3)
    transport_operator                = TransportOperatorMultipleAnglesVmap(TransportOperatorBatched(MGOP.transport_operator.solution_domain_data, MGOP.transport_operator.topological_sorted_angle, 0, 100))
    MGOP                              = eqx.tree_at(lambda t : t.transport_operator, MGOP, transport_operator)
    return MGOP, source_eg0, solution_domain

def setup_mgop_mod_theta(dim, degree, xs_data_indices, dx):
    MGOP, source_eg0, solution_domain = setup_mgop(dim, degree)    

    total_xs = MGOP.transport_operator.solution_domain_data.domain.cross_sections_indexed.cross_section_data.total
    scattering_xs = MGOP.transport_operator.solution_domain_data.domain.cross_sections_indexed.cross_section_data.scattering    
    if xs_data_indices[0] == 'total':
        total_xs = total_xs.at[xs_data_indices[1]].add(dx)
        
    elif xs_data_indices[0] == 'scattering':
        scattering_xs = scattering_xs.at[xs_data_indices[1]].add(dx)
    else:
        raise ValueError("Invalid xs_data_indices")    
    
    sgop = MGOP.to_single_group_operator(0)

    xs_data_new = CrossSectionData(total_xs, scattering_xs)
    sgop_new = eqx.tree_at(lambda sgop : [sgop.transport_operator.solution_domain_data.domain.cross_sections_indexed.cross_section_data, sgop.inscatter_operator.cross_sections_indexed.cross_section_data], sgop,[xs_data_new,xs_data_new])
    return sgop_new, source_eg0, solution_domain


MGOP, source_eg0, solution_domain = setup_mgop(2,2)

In [9]:
x_in = onp.random.rand(*MGOP.scatter_operator.D_in_structure().shape)
y_in = onp.random.rand(*MGOP.scatter_operator.D_out_structure().shape)
onp.testing.assert_equal( onp.sum(y_in * MGOP.scatter_operator.D_mv(x_in)) , onp.sum( MGOP.scatter_operator.D_transpose_mv(y_in) * x_in )  )

In [ ]:

from jax_sn.operators.moment_operator import MomentOperatorTransposed, MomentOperator
from jax_sn.operators.transport_operator import TransportOperatorMultipleAngles
from jax_sn.operators.mass_operator import MassOperator
from jax_sn.operators.scatter_operator import TotalGroupScatterOperator

from jax_sn.operators.scatter_operator.scatter_operator_util import _total_scattering_kernel_xs_indexed_multigroup_backwards

@eqx.filter_jit
def _compute_zeta_g_scan(n_energy_groups : int, moment_operator_t : MomentOperatorTransposed, transport_operator_t : TransportOperatorMultipleAngles, mass_operator_t : MassOperator, scatter_operator : TotalGroupScatterOperator, x : jnp.ndarray) -> jnp.ndarray:
    
    in_structure_shape = (n_energy_groups,) + moment_operator_t.in_structure().shape
    
    assert x.shape == in_structure_shape, f"Input x shape {x.shape} does not match expected in_structure shape {in_structure_shape}."
    result             = jnp.zeros(in_structure_shape, dtype=moment_operator_t.in_structure().dtype)
            
    def _single_energy_group(_, g_i):        
        return None, scatter_operator.D_transpose_mv(mass_operator_t.mv(transport_operator_t.to_energy_group(g_i).inv(moment_operator_t.mv(x[g_i]))))
            
    result = jax.lax.scan(_single_energy_group, None, jnp.arange(n_energy_groups))[1]
    return result


@eqx.filter_jit
def _multi_group_operator_transpose_scan(n_energy_groups : int, moment_operator    : MomentOperator, transport_operator : TransportOperatorMultipleAngles,  mass_operator      : MassOperator,  scatter_operator   : TotalGroupScatterOperator, x : jnp.ndarray) -> jnp.ndarray:    
    
    moment_operator_t    = moment_operator.transpose()
    transport_operator_t = transport_operator.transpose()
    mass_operator_t      = mass_operator.transpose()

    in_structure_shape   = (n_energy_groups,) + moment_operator_t.in_structure().shape
    assert x.shape == in_structure_shape, f"Input x shape {x.shape} does not match expected in_structure shape {in_structure_shape}."

    zeta_g_stored = _compute_zeta_g_scan(n_energy_groups, moment_operator_t, transport_operator_t, mass_operator_t, scatter_operator, x)

    def _scan_body(_, g_i):                
        return None, x[g_i] - _total_scattering_kernel_xs_indexed_multigroup_backwards(g_i, scatter_operator.cross_sections_indexed, zeta_g_stored)

    _, result = jax.lax.scan(_scan_body, None, jnp.arange(n_energy_groups))
    return result



In [35]:
onp.random.seed(12012026)
x = jnp.array( onp.random.rand(*MGOP.in_structure().shape))
y = jnp.array( onp.random.rand(*MGOP.out_structure().shape))

Mx = MGOP.mv(x)

In [30]:
MTy = _multi_group_operator_transpose_scan(MGOP.n_energy_groups, MGOP.moment_operator, MGOP.transport_operator, MGOP.mass_operator, MGOP.scatter_operator, y)

In [32]:
%timeit Mx = MGOP.mv(x).block_until_ready()

248 ms ± 9.42 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [33]:
%timeit MTy = _multi_group_operator_transpose_scan(MGOP.n_energy_groups, MGOP.moment_operator, MGOP.transport_operator, MGOP.mass_operator, MGOP.scatter_operator, y).block_until_ready()

241 ms ± 5.59 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
jnp.sum(Mx * y) , jnp.sum(x * MTy )

(Array(100139.47950699, dtype=float64), Array(100139.47950699, dtype=float64))

In [71]:
from jax_sn.domain.cross_sections import CrossSectionIndexed
from jax_sn.operators.base_linear_operator import AbstractLinearOperator_sn

class ScatterScaling(AbstractLinearOperator_sn):
    r'''
    Implements the scatter scaling operator:

    Mathematical Description
    ---------------------------
    .. math::
        \mathbf{S}_{g\tilde{g}} = \overline{\mathbf{S}}_g \otimes \mathbf{I}_B : \mathcal{M} \otimes \mathcal{F} \to  \mathcal{M} \otimes \mathcal{F}
        
    where :math:`\overline{\mathbf{S}}_g` is the single-element gathering scattering operator defined as:

    .. math::
        \overline{\mathbf{S}}_g[\psi_{\ell e}] = s_{\ell e} = \sum_{\overline{g}} \sigma_{\ell g \overline{g}e} \psi_{\overline{g}\ell e}

    '''
    cross_sections_indexed        : CrossSectionIndexed
    '''The indexed cross sections to use.'''
    energy_group_out              : int
    '''The energy group where neutrons are scattered to.'''
    energy_group_in               : int
    '''The energy group where neutrons are scattered from.'''
    elements_basis_shape         : jax.ShapeDtypeStruct  #[n_elements, n_basis]
    '''The shape of the elements and basis functions over which this operator is batched. Identified with :math:`\mathcal{F}` in the documentation above. However, note that the input and output are as [n_moments, n_elements, n_basis].'''

    def __check_init__(self):
        assert self.elements_basis_shape.ndim == 2, f"elements_basis_shape must be 2D [n_elements, n_basis], but has shape {self.elements_basis_shape.shape}."
        assert self.elements_basis_shape.shape[0] == self.cross_sections_indexed.indices.shape[0], f"Number of elements in elements_basis_shape {self.elements_basis_shape.shape[0]} must match number of elements in cross_sections_indexed {self.cross_sections_indexed.indices.shape[0]}."

    def mv(self, x : jnp.ndarray) -> jnp.ndarray:
        


<>:26: SyntaxWarning: invalid escape sequence '\m'
<>:26: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_34203/3782185856.py:26: SyntaxWarning: invalid escape sequence '\m'
  '''The shape of the elements and basis functions over which this operator is batched. Identified with :math:`\mathcal{F}` in the documentation above. However, note that the input and output are as [n_moments, n_elements, n_basis].'''


SyntaxError: incomplete input (3782185856.py, line 33)

In [53]:
_ = MGOP.mv(jnp.ones((MGOP.n_energy_groups,) + MGOP.moment_operator.out_structure().shape))

In [59]:
%timeit _compute_xi_g_map(MGOP.n_energy_groups,MGOP.moment_operator.transpose(), MGOP.transport_operator.transpose(), MGOP.mass_operator.transpose(), jnp.ones((MGOP.n_energy_groups,) + MGOP.moment_operator.out_structure().shape), batch_size=None).block_until_ready()


97.9 ms ± 398 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
